# Test: Créer un ticket avec user_token et user_id manuel

Cette fonction teste la création d'un ticket GLPI en passant directement l'`user_id` en paramètre (sans le récupérer dans la table). Utile pour:
- Tester avec des tokens utilisateur
- Assigner directement un user sans lookup
- Valider la liaison ticket-utilisateur

In [24]:
import requests
import json

# Configuration GLPI
GLPI_API_URL = "http://localhost/glpi/apirest.php"
APP_TOKEN = "uc00DWibqqnnfKd4mzJ5enb2dy9OlP9g6Xk7i0TG"
ADMIN_USER_TOKEN = "lKoBkb0nVBfN78FzxcbKemYKIUQKPHDoBEmhRqmB"  # Token admin
TARGET_USER_ID = 4  # L'utilisateur qu'on veut assigner comme requester


In [25]:
def init_session():
    auth_headers = {
        "Content-Type": "application/json",
        "Authorization": f"user_token {ADMIN_USER_TOKEN}",
        "App-Token": APP_TOKEN
    }
    response = requests.get(f"{GLPI_API_URL}/initSession", headers=auth_headers)
    if response.status_code != 200:
        print(f"Erreur authentification: {response.status_code} - {response.text}")
        return None
    session_token = response.json().get("session_token")
    print(f"Authentifie! Session: {session_token[:20]}...")
    return session_token

def get_headers(session_token):
    return {
        "Content-Type": "application/json",
        "Session-Token": session_token,
        "App-Token": APP_TOKEN
    }

def kill_session(session_token):
    requests.get(f"{GLPI_API_URL}/killSession", headers=get_headers(session_token))
    print("Session fermee")


In [26]:
def test_approche1_requester_in_creation():
    session_token = init_session()
    if not session_token:
        return None

    headers = get_headers(session_token)

    ticket_payload = {
        "input": {
            "name": "Test - Requester via _users_id_requester",
            "content": "Ticket cree par admin, requester = user_id 5",
            "itilcategories_id": 5,
            "priority": 3,
            "urgency": 3,
            "impact": 3,
            "_users_id_requester": TARGET_USER_ID  # <-- la cle
        }
    }

    print(f"\nCreation du ticket avec _users_id_requester={TARGET_USER_ID}...")
    response = requests.post(f"{GLPI_API_URL}/Ticket", headers=headers, json=ticket_payload)

    if response.status_code not in [200, 201]:
        print(f"Erreur creation: {response.status_code} - {response.text}")
        kill_session(session_token)
        return None

    ticket_id = response.json().get("id")
    print(f"Ticket cree: ID={ticket_id}")

    # Verifier via Ticket_User
    users_response = requests.get(f"{GLPI_API_URL}/Ticket/{ticket_id}/Ticket_User", headers=headers)
    if users_response.status_code == 200:
        for u in users_response.json():
            type_label = {1: "Requester", 2: "Assign", 3: "Observer"}.get(u.get("type"), "Unknown")
            print(f"  - User ID: {u.get('users_id')}, Type: {type_label}")

    kill_session(session_token)
    return ticket_id

result_1 = test_approche1_requester_in_creation()


Authentifie! Session: 21dsnnv53lsv5oa9ernd...

Creation du ticket avec _users_id_requester=4...
Ticket cree: ID=94
  - User ID: 4, Type: Requester
Session fermee
